# Module 3 — Analytical Modeling

Homework: [cohorts/2026/homework3.md](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework3.md)
Submit: [courses.datatalks.club/sma-zoomcamp-2026/homework/hw03](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw03)

Kernel: **Python (stock-markets-zoomcamp)** — the shared repo venv.

| Q | Question | Answer |
| --- | --- | --- |
| 1 | Max abs. correlation of a `<month>_w<week>` dummy with `is_positive_growth_30d_future` | **0.025** (`month_wom_October_w4`) |
| 2 | Precision of the best new hand rule (`pred3` / `pred4`) | **0.580** (`pred3_manual_dgs10_5`) |
| 3 | TEST records where only `pred5_clf_10` is correct | **3,802** |
| 4 | Optimal `max_depth` for the `DecisionTreeClassifier` | **5** (test precision 0.628) |
| 5 | What data is missing? | measured experiment — top-decile precision **0.551 → 0.708** (RandomForest + engineered blocks) |

This notebook reproduces the dataset from the lecture Colab
([`[2025]_Module_3_Colab_Time_Series_Modeling.ipynb`](./%5B2025%5D_Module_3_Colab_Time_Series_Modeling.ipynb))
rather than rebuilding features from scratch, so the numbers line up with the ones quoted
in the task. The source file is the same Google Drive parquet the Colab `gdown`s —
33 tickers (US / EU / India), truncated to 2000-01-01 onwards.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score

import smaz
from smaz import config, utils

utils.set_plot_defaults()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print("smaz", smaz.__version__, "| repo root:", config.REPO_ROOT)

## 0. The lecture dataset

Same file as the Colab: `stocks_df_combined_2025_06_13.parquet.brotli`, pulled once from
Google Drive into the shared (gitignored) `data/raw/` cache and reused on every re-run.

In [ ]:
COURSE_PARQUET = config.RAW_DIR / "stocks_df_combined_2025_06_13.parquet.brotli"
GDRIVE_ID = "1mb0ae2M5AouSDlqcUnIwaHq7avwGNrmB"  # the id the lecture Colab gdowns

if not COURSE_PARQUET.exists():
    import gdown

    gdown.download(id=GDRIVE_ID, output=str(COURSE_PARQUET))

df_full = pd.read_parquet(COURSE_PARQUET)
print(df_full.shape, "|", df_full.Ticker.nunique(), "tickers |",
      df_full.Date.min().date(), "->", df_full.Date.max().date())

### 0.1 Feature groups

Identical to the Colab's section 0, with `month_wom` added to `CATEGORICAL` up front (Q1).

In [ ]:
# growth indicators (but not future growth)
GROWTH = [g for g in df_full.keys() if (g.find("growth_") == 0) & (g.find("future") < 0)]

OHLCV = ["Open", "High", "Low", "Close", "Adj Close_x", "Volume"]

# 'month_wom' is the new Q1 feature; everything else is as in the lecture
CATEGORICAL = ["Month", "Weekday", "Ticker", "ticker_type", "month_wom"]

TO_PREDICT = [g for g in df_full.keys() if g.find("future") >= 0]
TO_DROP = ["Year", "Date", "index_x", "index_y", "index", "Quarter", "Adj Close_y"] + CATEGORICAL + OHLCV

# one custom numerical feature on top of the parquet.
# The lecture writes np.log(Volume), which is -inf on zero-volume (halted) days. Mapping those
# to NaN instead is identical for Q1-Q4 (both become 0 in clean_features) but keeps the rolling
# z-scores in Q5's Block C finite -- a single -inf poisons every window it touches.
df_full["ln_volume"] = np.log(df_full.Volume.replace(0, np.nan))

CUSTOM_NUMERICAL = [
    "SMA10", "SMA20", "growing_moving_average", "high_minus_low_relative", "volatility", "ln_volume",
]

TECHNICAL_INDICATORS = [
    "adx", "adxr", "apo", "aroon_1", "aroon_2", "aroonosc", "bop", "cci", "cmo", "dx",
    "macd", "macdsignal", "macdhist", "macd_ext", "macdsignal_ext", "macdhist_ext",
    "macd_fix", "macdsignal_fix", "macdhist_fix", "mfi", "minus_di", "mom", "plus_di",
    "dm", "ppo", "roc", "rocp", "rocr", "rocr100", "rsi", "slowk", "slowd", "fastk",
    "fastd", "fastk_rsi", "fastd_rsi", "trix", "ultosc", "willr", "ad", "adosc", "obv",
    "atr", "natr", "ht_dcperiod", "ht_dcphase", "ht_phasor_inphase", "ht_phasor_quadrature",
    "ht_sine_sine", "ht_sine_leadsine", "ht_trendmod", "avgprice", "medprice", "typprice", "wclprice",
]

TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find("cdl") >= 0]

MACRO = [
    "gdppot_us_yoy", "gdppot_us_qoq", "cpi_core_yoy", "cpi_core_mom",
    "FEDFUNDS", "DGS1", "DGS5", "DGS10",
]

NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO

print(f"{len(GROWTH)} growth, {len(TECHNICAL_INDICATORS)} indicators, "
      f"{len(TECHNICAL_PATTERNS)} candle patterns, {len(MACRO)} macro -> {len(NUMERICAL)} numerical")
print("to predict:", TO_PREDICT)

In [ ]:
# same truncation as the lecture: ~25 years, so the growth features are all populated
df = df_full[df_full.Date >= "2000-01-01"].copy()
print(df.shape)
df.groupby("Ticker")["Date"].agg(["min", "max", "count"]).head()

---
## Question 1 — Dummies for Month and Week-of-Month

> **What is the ABSOLUTE CORRELATION VALUE of the most correlated dummy variable
> `<month>_w<week_of_month>` with the binary outcome `is_positive_growth_30d_future`?**
> Round to three decimals.

In [ ]:
# Month/Weekday to strings, exactly as the lecture does before get_dummies()
df["Month"] = df.Month.dt.strftime("%B")
df["Weekday"] = df.Weekday.astype(str)

# week of month: (day - 1) // 7 + 1  -> 1..5
df["week_of_month"] = (df.Date.dt.day - 1) // 7 + 1
df["month_wom"] = df.Month + "_w" + df.week_of_month.astype(str)

df[["Date", "Month", "Weekday", "week_of_month", "month_wom"]].head()

In [ ]:
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype="int32")
DUMMIES = dummy_variables.keys().to_list()

WOM_DUMMIES = [d for d in DUMMIES if d.startswith("month_wom_")]
print(f"{len(DUMMIES)} dummies in total, {len(WOM_DUMMIES)} of them from month_wom")

df_with_dummies = pd.concat([df, dummy_variables], axis=1)

In [ ]:
corr_target = df_with_dummies[NUMERICAL + DUMMIES + TO_PREDICT].corr()["is_positive_growth_30d_future"]

corr_wom = corr_target[WOM_DUMMIES].to_frame()
corr_wom["abs_corr"] = corr_wom["is_positive_growth_30d_future"].abs()
corr_wom = corr_wom.sort_values("abs_corr", ascending=False)

corr_wom.head(10)

In [ ]:
q1_feature = corr_wom.index[0]
q1_answer = round(corr_wom.abs_corr.iloc[0], 3)
print(f"most correlated month_wom dummy: {q1_feature} -> {corr_wom.abs_corr.iloc[0]:.5f} -> {q1_answer}")

**Answer:** **0.025** — `month_wom_October_w4`.

The strongest week-of-month dummy is the **fourth week of October** — consistent with the
hint that October/November carry seasonality. Note how weak the signal is in absolute terms:
|corr| = 0.025 means the dummy explains ~0.06% of the variance of the target, so it is a
tie-breaker for the tree at best, not a tradeable rule on its own. The negative side of the
table is the mirror image — the first weeks of January/February are the worst-correlated.

---
## Temporal split (70 / 15 / 15 by date)

Straight from the lecture's code snippet 3 — needed by every question below.

In [ ]:
def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    """Label every row train/validation/test by calendar date, not by row count."""
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)

    df["split"] = np.select(
        [df["Date"] <= train_end, df["Date"] <= val_end],
        ["train", "validation"],
        default="test",
    )
    return df


df_with_dummies = temporal_split(
    df_with_dummies,
    min_date=df_with_dummies.Date.min(),
    max_date=df_with_dummies.Date.max(),
)

# copy once to drop the post-join fragmentation warning
new_df = df_with_dummies.copy()

new_df.groupby("split")["Date"].agg(["min", "max", "count"]).assign(
    share=lambda d: (d["count"] / len(new_df)).round(3)
)

---
## Question 2 — New "hand" rules on macro indicators

> **What is the precision score for the best of the NEW predictions (`pred3` or `pred4`),
> rounded to 3 digits after the comma?**

In [ ]:
# the three rules from the lecture
new_df["pred0_manual_cci"] = (new_df.cci > 200).astype(int)
new_df["pred1_manual_prev_g1"] = (new_df.growth_30d > 1).astype(int)
new_df["pred2_manual_prev_g1_and_snp"] = ((new_df.growth_30d > 1) & (new_df.growth_snp500_30d > 1)).astype(int)

# the two new ones, read off the clf10 tree branches that lead to a 'positive' leaf
new_df["pred3_manual_dgs10_5"] = ((new_df.DGS10 <= 4) & (new_df.DGS5 <= 1)).astype(int)
new_df["pred4_manual_dgs10_fedfunds"] = ((new_df.DGS10 > 4) & (new_df.FEDFUNDS <= 4.795)).astype(int)

PREDICTIONS = [k for k in new_df.keys() if k.startswith("pred")]
PREDICTIONS

In [ ]:
def add_is_correct(df, prediction_columns, target="is_positive_growth_30d_future"):
    """is_correct_predN == 1 when predN matched the realised direction."""
    for pred in prediction_columns:
        prefix = pred.split("_")[0]
        df[f"is_correct_{prefix}"] = (df[pred] == df[target]).astype(int)
    return [f"is_correct_{p.split('_')[0]}" for p in prediction_columns]


IS_CORRECT = add_is_correct(new_df, PREDICTIONS)
new_df[PREDICTIONS + IS_CORRECT + ["is_positive_growth_30d_future"]].sample(5, random_state=42)

In [ ]:
def precision_table(df, prediction_columns, split="test"):
    """TP / (TP + FP) for every rule, counted only where the rule says '1'."""
    rows = []
    for pred in prediction_columns:
        prefix = pred.split("_")[0]
        mask = (df.split == split) & (df[pred] == 1)
        n_positive = int(mask.sum())
        rows.append(
            {
                "prediction": pred,
                "positive_calls": n_positive,
                "true_positives": int(df.loc[mask, f"is_correct_{prefix}"].sum()),
                "precision": df.loc[mask, f"is_correct_{prefix}"].mean() if n_positive else np.nan,
            }
        )
    return pd.DataFrame(rows).set_index("prediction")


prec_test = precision_table(new_df, PREDICTIONS)
prec_test.round(4)

In [ ]:
q2_answer = round(prec_test.loc[["pred3_manual_dgs10_5", "pred4_manual_dgs10_fedfunds"], "precision"].max(), 3)
print("pred3 positive calls on TEST:", int(prec_test.loc["pred3_manual_dgs10_5", "positive_calls"]))
print("pred4 positive calls on TEST:", int(prec_test.loc["pred4_manual_dgs10_fedfunds", "positive_calls"]))
print("best new rule precision:", q2_answer)

**Answer:** **0.580** — `pred3_manual_dgs10_5` — 997 positive calls on TEST, 578 of them correct.

`pred4_manual_dgs10_fedfunds` fires far more often (5,660 calls) but only 0.466 precision,
which is *below* the 0.551 base rate of `is_positive_growth_30d_future` on TEST — a rule
worse than always calling "up".

Why the thresholds had to be relaxed from the tree's own splits: the TEST window is
2021-08-20 → 2025-06-13, where `DGS5` never drops below 0.77 and `FEDFUNDS` is at 5.33
whenever `DGS10` exceeds 4.825. Both original conditions therefore produce **zero** positive
calls on TEST, and precision is undefined. The tree learned those splits on the ZIRP years
that sit in TRAIN/VALIDATION; rounding the cut-points outward (`DGS5 <= 1`, `DGS10 <= 4`)
brings the rule back into the range the TEST period actually visits.

---
## Question 3 — Unique correct predictions from `pred5_clf_10`

> **How many TEST records does `pred5_clf_10` get right while every hand rule
> (`pred0` … `pred4`) gets them wrong?**

In [ ]:
FEATURES = NUMERICAL + DUMMIES
TO_PREDICT_COL = "is_positive_growth_30d_future"


def clean_features(X):
    """Trees can't take +-inf (ln_volume when Volume == 0); NaNs become 0, as in the lecture."""
    return X.replace([np.inf, -np.inf], np.nan).fillna(0)


train_mask = new_df.split.isin(["train", "validation"]).to_numpy()
test_mask = (new_df.split == "test").to_numpy()

X_all = clean_features(new_df[FEATURES])
y_all = new_df[TO_PREDICT_COL]

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

print(f"X_train {X_train.shape}, X_test {X_test.shape}, X_all {X_all.shape}")

In [ ]:
%%time
clf_10 = DecisionTreeClassifier(max_depth=10, random_state=42).fit(X_train, y_train)

# predict on the FULL frame: same length and order, so this is a plain column assignment
new_df["pred5_clf_10"] = clf_10.predict(X_all)
new_df["is_correct_pred5"] = (new_df.pred5_clf_10 == new_df[TO_PREDICT_COL]).astype(int)

print("accuracy (test) :", round(accuracy_score(y_test, new_df.loc[test_mask, "pred5_clf_10"]), 4))
print("precision (test):", round(precision_score(y_test, new_df.loc[test_mask, "pred5_clf_10"]), 4))

In [ ]:
# top of the tree — this is where the pred3/pred4 rules in Q2 come from
print(export_text(clf_10, feature_names=list(X_train.columns), max_depth=2))

In [ ]:
feature_importance = (
    pd.DataFrame({"feature": X_train.columns, "importance": clf_10.feature_importances_})
    .sort_values("importance", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
feature_importance

In [ ]:
def only_this_is_correct(df, target_prefix, all_prefixes):
    """True where `target_prefix` is right and every other predictor is wrong.

    Written against the is_correct_* columns so it scales to pred0..pred99 without
    hard-coding a condition per rule.
    """
    others = [f"is_correct_{p}" for p in all_prefixes if p != target_prefix]
    return (df[f"is_correct_{target_prefix}"] == 1) & (df[others].sum(axis=1) == 0)


HAND_PREFIXES = ["pred0", "pred1", "pred2", "pred3", "pred4"]
new_df["only_pred5_is_correct"] = only_this_is_correct(new_df, "pred5", HAND_PREFIXES + ["pred5"]).astype(int)

q3_answer = int(new_df.loc[test_mask, "only_pred5_is_correct"].sum())
print("records on TEST where only pred5_clf_10 is correct:", q3_answer)

**Answer:** **3,802** records.

Roughly 12% of the TEST set — the tree is not just re-deriving the hand rules, it finds a
meaningful slice of days that none of `pred0`…`pred4` gets right. (It also *loses* days the
hand rules win; the count above is one-directional by construction.)

### Advanced (optional) — generalising to `pred0` … `predN`

The vectorised helper above is enough for one predictor. The task asks for the form that
scales to an arbitrary number of prediction columns without hard-coding a condition per rule,
written as a function
[applied to every row](https://sparkbyexamples.com/pandas/pandas-apply-function-to-every-row/).

In [ ]:
def uniquely_correct_row(row, target_prefix, all_prefixes):
    """1 when `target_prefix` got this row right and every other predictor got it wrong.

    Row-wise, so it works unchanged for pred0..pred99 — at the cost of one Python call per
    row, which is why the vectorised version above is what the rest of the notebook uses.
    """
    correct = {p: row[f"is_correct_{p}"] == 1 for p in all_prefixes}
    others = [v for p, v in correct.items() if p != target_prefix]
    return int(correct[target_prefix] and not any(others))


ALL_PREFIXES = HAND_PREFIXES + ["pred5"]
test_rows = new_df.loc[test_mask, [f"is_correct_{p}" for p in ALL_PREFIXES]]

row_wise_count = int(test_rows.apply(uniquely_correct_row, axis=1, args=("pred5", ALL_PREFIXES)).sum())
print("row-wise  :", row_wise_count)
print("vectorised:", q3_answer)
assert row_wise_count == q3_answer, "row-wise and vectorised must agree"

In [ ]:
def unique_correctness_table(df, prefixes, mask):
    """For every predictor: how often is it right, and how often is it the ONLY one right?"""
    sub = df.loc[mask, [f"is_correct_{p}" for p in prefixes]]
    n_correct = sub.sum(axis=1)
    out = pd.DataFrame(
        {
            "correct": [int(sub[f"is_correct_{p}"].sum()) for p in prefixes],
            "uniquely_correct": [
                int(((sub[f"is_correct_{p}"] == 1) & (n_correct == 1)).sum()) for p in prefixes
            ],
        },
        index=prefixes,
    )
    return out.assign(unique_share=lambda d: (d.uniquely_correct / d.correct).round(3))


unique_correctness_table(new_df, ALL_PREFIXES, test_mask)

---
## Question 4 — Hyperparameter tuning (tree depth)

> **What is the optimal tree depth (1 … 20) for a `DecisionTreeClassifier`?**

Trained on TRAIN+VALIDATION, scored by precision on TEST (the validation column is
tracked alongside just to see where the tree starts memorising).

In [ ]:
%%time
val_mask = (new_df.split == "validation").to_numpy()
X_val, y_val = X_all[val_mask], y_all[val_mask]

rows = []
for depth in range(1, 21):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    pred_train, pred_val, pred_test = clf.predict(X_train), clf.predict(X_val), clf.predict(X_test)
    rows.append(
        {
            "max_depth": depth,
            "precision_test": precision_score(y_test, pred_test, zero_division=0),
            "precision_val": precision_score(y_val, pred_val, zero_division=0),
            "precision_train": precision_score(y_train, pred_train, zero_division=0),
            "accuracy_test": accuracy_score(y_test, pred_test),
            "accuracy_val": accuracy_score(y_val, pred_val),
            "accuracy_train": accuracy_score(y_train, pred_train),
            "auc_test": roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]),
            "n_leaves": clf.get_n_leaves(),
            "positive_calls_test": int((pred_test == 1).sum()),
        }
    )

depth_scan = pd.DataFrame(rows).set_index("max_depth")
depth_scan[["precision_test", "precision_val", "auc_test", "n_leaves", "positive_calls_test"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(depth_scan.index, depth_scan.precision_test, marker="o", label="TEST")
ax.plot(depth_scan.index, depth_scan.precision_val, marker="o", alpha=0.5, label="VALIDATION (in-sample)")

best_max_depth = int(depth_scan.precision_test.idxmax())
ax.axvline(best_max_depth, color="grey", ls="--", lw=1)
ax.annotate(
    f"best depth = {best_max_depth}",
    xy=(best_max_depth, depth_scan.precision_test.max()),
    xytext=(6, 8), textcoords="offset points",
)

ax.set_xticks(range(1, 21))
ax.set_xlabel("max_depth")
ax.set_ylabel("precision")
ax.set_title("Decision tree precision vs. depth (trained on TRAIN+VALIDATION)")
ax.legend()
plt.show()

print("best_max_depth =", best_max_depth,
      "| test precision =", round(depth_scan.precision_test.max(), 4))

In [ ]:
%%time
clf_best = DecisionTreeClassifier(max_depth=best_max_depth, random_state=42).fit(X_train, y_train)
new_df["pred6_clf_best"] = clf_best.predict(X_all)

ALL_PREDICTIONS = PREDICTIONS + ["pred5_clf_10", "pred6_clf_best"]
IS_CORRECT = add_is_correct(new_df, ALL_PREDICTIONS)

precision_table(new_df, ALL_PREDICTIONS).round(4)

**Answer:** **5** — test precision **0.628**, comfortably above the 0.58 target and above every
hand rule as well as the depth-10 tree (0.589).

The curve is not monotonic: depths 1–4 are effectively "always predict up" (~31k positive
calls out of 31,408 TEST rows, precision ≈ the base rate), depth 5 is the sweet spot where
the tree becomes selective (18,452 calls) without yet memorising, and from depth 6 onward
precision oscillates in the 0.57–0.60 band while the validation curve keeps climbing —
the classic overfitting signature.

### Advanced (optional) — saturation, overfitting and the complexity trade-off

Precision *and* accuracy, on all three splits, against `max_depth` — plus the leaf count as
the direct measure of model complexity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, metric, title in [
    (axes[0], "precision", "Precision"),
    (axes[1], "accuracy", "Accuracy"),
]:
    for split, style in [("train", "-"), ("val", "--"), ("test", "-")]:
        ax.plot(depth_scan.index, depth_scan[f"{metric}_{split}"], style, marker="o",
                ms=3, label=split.upper())
    ax.axvline(best_max_depth, color="grey", ls=":", lw=1)
    ax.set_xticks(range(0, 21, 2))
    ax.set_xlabel("max_depth")
    ax.set_title(f"{title} vs. tree depth")
    ax.legend(fontsize=8)

ax2 = axes[1].twinx()
ax2.plot(depth_scan.index, depth_scan.n_leaves, color="tab:red", alpha=0.35, lw=2)
ax2.set_yscale("log")
ax2.set_ylabel("leaves (log)", color="tab:red")

plt.tight_layout()
plt.show()

In [ ]:
# complexity vs. generalisation, in numbers
complexity = depth_scan.assign(
    train_test_gap=lambda d: (d.accuracy_train - d.accuracy_test).round(4),
    leaves_per_1k_rows=lambda d: (d.n_leaves / (len(X_train) / 1000)).round(2),
)[["n_leaves", "leaves_per_1k_rows", "accuracy_train", "accuracy_test",
   "train_test_gap", "precision_test", "auc_test"]]
complexity.round(4)

The optional tree-head visualisation: how the top two levels change as the tree is allowed
to grow. The root split is stable, what changes underneath it is where the extra capacity goes.

In [ ]:
for depth in (1, 2, 3, best_max_depth):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    print(f"{'=' * 20} max_depth = {depth}  ({clf.get_n_leaves()} leaves) {'=' * 20}")
    print(export_text(clf, feature_names=list(X_train.columns), max_depth=2))

In [ ]:
plt.figure(figsize=(18, 7))
plot_tree(
    DecisionTreeClassifier(max_depth=best_max_depth, random_state=42).fit(X_train, y_train),
    filled=True,
    feature_names=list(X_train.columns),
    class_names=["Negative", "Positive"],
    max_depth=2,
    fontsize=8,
)
plt.title(f"Top 2 levels of the tuned tree (max_depth={best_max_depth})")
plt.show()

---
## Question 5 — What data is missing?

Rather than list features I *think* would help, this section builds them and measures whether
they do. Four blocks are engineered, each aimed at a specific blind spot visible in the Q4
model, and each is evaluated on TEST by **precision** and **ROC AUC** against the homework
feature set.

**Constraint:** everything below is derived from the parquet already on disk — no new
downloads, no external APIs, no network. That rules out the obvious wish-list items (VIX,
credit spreads, ECB/RBI rates, earnings dates), so where a block needs a market-wide risk
signal it is *reconstructed* from index growth columns that are already in the frame.

### The blind spots

| # | Blind spot in the homework feature set | Block |
| --- | --- | --- |
| A | Rates appear only as **levels** (`DGS1/5/10`, `FEDFUNDS`). No curve shape, no real rate, no direction of travel — yet rates dominate the tree's importances. | macro curve & policy dynamics |
| B | Every feature is **absolute and per-ticker**. Nothing says whether this stock is strong *relative to its peers today*. | cross-sectional relative strength |
| C | No feature says **where the price sits in its own recent range** (52-week position, distance from its own moving averages, whether volume/vol is unusual *for this ticker*). | own-history position |
| D | No **market regime** signal: a 3% drop in a calm tape and in a drawdown look identical. | reconstructed market regime |

### Block A — macro curve and policy dynamics

The macro columns repeat across tickers, so these are computed once on the unique-date series
and merged back — diffs taken per ticker would be wrong wherever calendars differ.

In [ ]:
def add_macro_dynamics(df):
    """Curve shape, real rates, policy gap and rate momentum from the existing DGS/FEDFUNDS levels."""
    m = (
        df[["Date", "DGS1", "DGS5", "DGS10", "FEDFUNDS", "cpi_core_yoy"]]
        .drop_duplicates("Date")
        .sort_values("Date")
        .set_index("Date")
    )
    out = pd.DataFrame(index=m.index)

    # curve shape
    out["term_10y_1y"] = m.DGS10 - m.DGS1
    out["term_10y_5y"] = m.DGS10 - m.DGS5
    out["curve_5y_1y"] = m.DGS5 - m.DGS1

    # inflation-adjusted levels
    out["real_10y"] = m.DGS10 - m.cpi_core_yoy
    out["real_fedfunds"] = m.FEDFUNDS - m.cpi_core_yoy

    # what the 1y bill prices in vs. where policy actually is -> cut/hike expectation
    out["policy_gap_1y"] = m.DGS1 - m.FEDFUNDS

    # direction of travel (trading days: 21 ~ 1m, 63 ~ 3m, 126 ~ 6m)
    for col, lags in [("DGS10", [21, 63, 126]), ("DGS1", [21, 63]),
                      ("FEDFUNDS", [63, 126]), ("cpi_core_yoy", [126])]:
        for lag in lags:
            out[f"{col.lower()}_chg_{lag}d"] = m[col].diff(lag)

    out["is_curve_inverted"] = (out.term_10y_1y < 0).astype(int)
    out["is_cutting_cycle"] = (out.fedfunds_chg_63d < -0.1).astype(int)
    out["is_hiking_cycle"] = (out.fedfunds_chg_63d > 0.1).astype(int)

    return df.merge(out.reset_index(), on="Date", how="left"), list(out.columns)

### Block B — cross-sectional relative strength

Percentile rank *within each date*, across all 33 tickers and again within the ticker's own
region, plus excess growth over the S&P and over the ticker's home index
(US → S&P 500, EU → DAX, India → EPI). Only same-day information is used, so there is no leak.

In [ ]:
HOME_INDEX = {"US": "growth_snp500_", "EU": "growth_dax_", "INDIA": "growth_epi_"}


def add_cross_sectional(df):
    """Where does this ticker stand against every other ticker *on this date*?"""
    cols = []
    for col in ["growth_30d", "growth_90d", "growth_365d", "rsi", "volatility", "natr", "ln_volume", "mom"]:
        df[f"csrank_{col}"] = df.groupby("Date")[col].rank(pct=True)
        df[f"csrank_reg_{col}"] = df.groupby(["Date", "ticker_type"])[col].rank(pct=True)
        cols += [f"csrank_{col}", f"csrank_reg_{col}"]

    for horizon in ["30d", "90d", "365d"]:
        df[f"excess_snp_{horizon}"] = df[f"growth_{horizon}"] - df[f"growth_snp500_{horizon}"]

        home = pd.Series(np.nan, index=df.index)
        for ticker_type, prefix in HOME_INDEX.items():
            rows = df.ticker_type == ticker_type
            home[rows] = df.loc[rows, f"{prefix}{horizon}"]
        df[f"excess_home_{horizon}"] = df[f"growth_{horizon}"] - home

        cols += [f"excess_snp_{horizon}", f"excess_home_{horizon}"]

    return df, cols

### Block C — position within the ticker's own history

Rolling windows are backward-looking (`rolling(...)` ends at the current row), so these are
point-in-time. `min_periods` keeps early rows NaN rather than fitting them on two data points.

In [ ]:
def add_own_history(df):
    """52-week position, distance from own moving averages, and self-normalised vol/volume.

    Requires df sorted by (Ticker, Date).
    """
    g = df.groupby("Ticker", group_keys=False)

    roll_max = g["Close_x"].transform(lambda s: s.rolling(252, min_periods=60).max())
    roll_min = g["Close_x"].transform(lambda s: s.rolling(252, min_periods=60).min())
    df["pct_from_52w_high"] = df.Close_x / roll_max - 1
    df["pct_from_52w_low"] = df.Close_x / roll_min - 1

    df["close_over_sma10"] = df.Close_x / df.SMA10
    df["close_over_sma20"] = df.Close_x / df.SMA20
    df["sma10_over_sma20"] = df.SMA10 / df.SMA20

    # z-score each ticker against its OWN recent history -> comparable across tickers and eras
    for col, window in [("volatility", 60), ("ln_volume", 20), ("rsi", 252), ("natr", 60)]:
        mu = g[col].transform(lambda s: s.rolling(window, min_periods=max(10, window // 4)).mean())
        sd = g[col].transform(lambda s: s.rolling(window, min_periods=max(10, window // 4)).std())
        df[f"{col}_z{window}"] = (df[col] - mu) / sd

    return df, [
        "pct_from_52w_high", "pct_from_52w_low", "close_over_sma10", "close_over_sma20",
        "sma10_over_sma20", "volatility_z60", "ln_volume_z20", "rsi_z252", "natr_z60",
    ]

### Block D — market regime, reconstructed from in-frame index growth

No VIX available without a download, so it gets rebuilt: `growth_snp500_1d` is the same for
every ticker on a date, so cumulative-producting the unique-date series recovers an index
*level*, and from a level one gets drawdown and realised volatility — a serviceable local
stand-in for "is the market calm or breaking?".

In [ ]:
def add_market_regime(df):
    """Drawdown and realised vol of each index, rebuilt from its daily growth column."""
    d = (
        df[["Date", "growth_snp500_1d", "growth_dax_1d", "growth_epi_1d",
            "growth_gold_90d", "growth_wti_oil_90d", "growth_btc_usd_30d"]]
        .drop_duplicates("Date")
        .sort_values("Date")
        .set_index("Date")
    )
    out = pd.DataFrame(index=d.index)

    for name, col in [("snp", "growth_snp500_1d"), ("dax", "growth_dax_1d"), ("epi", "growth_epi_1d")]:
        level = d[col].fillna(1).cumprod()  # daily growth -> synthetic index level
        out[f"{name}_dd_252d"] = level / level.rolling(252, min_periods=60).max() - 1
        daily_return = d[col] - 1
        out[f"{name}_rvol_21d"] = daily_return.rolling(21, min_periods=10).std() * np.sqrt(252)

    # the VIX stand-in: is realised vol high relative to its own trailing year?
    rvol = out.snp_rvol_21d
    out["mkt_rvol_z"] = (rvol - rvol.rolling(252, min_periods=60).mean()) / rvol.rolling(252, min_periods=60).std()

    out["gold_over_oil_90d"] = d.growth_gold_90d / d.growth_wti_oil_90d  # flight-to-safety proxy
    out["risk_on_btc_30d"] = d.growth_btc_usd_30d
    out["is_mkt_correction"] = (out.snp_dd_252d < -0.10).astype(int)

    return df.merge(out.reset_index(), on="Date", how="left"), list(out.columns)

In [ ]:
%%time
# isolated copy: the rolling features need (Ticker, Date) order, new_df must keep its own
exp_df = new_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

exp_df, BLOCK_A = add_macro_dynamics(exp_df)
exp_df, BLOCK_B = add_cross_sectional(exp_df)
exp_df, BLOCK_C = add_own_history(exp_df)
exp_df, BLOCK_D = add_market_regime(exp_df)

BASE_FEATURES = NUMERICAL + DUMMIES
print(f"BASE {len(BASE_FEATURES)} features"
      f" | A {len(BLOCK_A)} | B {len(BLOCK_B)} | C {len(BLOCK_C)} | D {len(BLOCK_D)}")

### Evaluation protocol

One fix relative to Q4: the homework picks `max_depth` by **TEST** precision, which is
selection on the test set. Here depth is chosen on **VALIDATION**, the model is refitted on
TRAIN+VALIDATION at that depth, and TEST is touched exactly once per feature set. The
resulting numbers are lower than Q4's 0.628 and that is the point — they are the honest ones.

Metrics: precision (the decision that gets traded), ROC AUC (ranking quality across the whole
probability range, insensitive to the 0.5 threshold), and positive-call count.

In [ ]:
exp_train = (exp_df.split == "train").to_numpy()
exp_val = (exp_df.split == "validation").to_numpy()
exp_test = (exp_df.split == "test").to_numpy()
exp_trainval = exp_train | exp_val
y_exp = exp_df[TO_PREDICT_COL]

DEPTH_GRID = [3, 4, 5, 6, 7, 8, 10, 12]


def evaluate_feature_set(name, feature_list):
    """Tune depth on VALIDATION, refit on TRAIN+VALIDATION, score once on TEST."""
    X = clean_features(exp_df[feature_list])

    best_depth, best_val = None, -1.0
    for depth in DEPTH_GRID:
        clf = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X[exp_train], y_exp[exp_train])
        val_precision = precision_score(y_exp[exp_val], clf.predict(X[exp_val]), zero_division=0)
        if val_precision > best_val:
            best_val, best_depth = val_precision, depth

    clf = DecisionTreeClassifier(max_depth=best_depth, random_state=42).fit(X[exp_trainval], y_exp[exp_trainval])
    pred = clf.predict(X[exp_test])
    proba = clf.predict_proba(X[exp_test])[:, 1]

    return {
        "feature_set": name,
        "n_features": len(feature_list),
        "best_depth": best_depth,
        "val_precision": round(best_val, 4),
        "test_precision": round(precision_score(y_exp[exp_test], pred, zero_division=0), 4),
        "test_auc": round(roc_auc_score(y_exp[exp_test], proba), 4),
        "positive_calls": int(pred.sum()),
    }

In [ ]:
%%time
ALL_FEATURES = BASE_FEATURES + BLOCK_A + BLOCK_B + BLOCK_C + BLOCK_D

experiments = [
    ("BASE (homework features)", BASE_FEATURES),
    ("BASE + A macro curve", BASE_FEATURES + BLOCK_A),
    ("BASE + B cross-sectional", BASE_FEATURES + BLOCK_B),
    ("BASE + C own-history", BASE_FEATURES + BLOCK_C),
    ("BASE + D market regime", BASE_FEATURES + BLOCK_D),
    ("BASE + ALL blocks", ALL_FEATURES),
]

block_results = pd.DataFrame([evaluate_feature_set(n, f) for n, f in experiments]).set_index("feature_set")
baseline = block_results.loc["BASE (homework features)"]
block_results.assign(
    d_precision=lambda d: (d.test_precision - baseline.test_precision).round(4),
    d_auc=lambda d: (d.test_auc - baseline.test_auc).round(4),
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
labels = [i.replace("BASE + ", "").replace("BASE ", "") for i in block_results.index]

for ax, col, ref, title in [
    (axes[0], "test_precision", baseline.test_precision, "TEST precision"),
    (axes[1], "test_auc", baseline.test_auc, "TEST ROC AUC"),
]:
    colors = ["tab:grey"] + ["tab:green" if v > ref else "tab:red" for v in block_results[col][1:]]
    ax.barh(labels, block_results[col], color=colors)
    ax.axvline(ref, color="black", ls="--", lw=1, label="baseline")
    ax.set_xlim(min(block_results[col]) * 0.97, max(block_results[col]) * 1.02)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("TEST base rate (always predict 'up'):", round(y_exp[exp_test].mean(), 4))

### Does a stronger model class help more than the features?

A single tree throws away most of the AUC available — it can only emit as many distinct
probabilities as it has leaves, and thresholding those at 0.5 discards the ranking entirely.
The grid below crosses **feature set** with **model class** to separate "the features carry no
signal" from "a single tree cannot use them", and adds **precision on the top decile of
predicted probability** — what you would actually earn trading only your strongest signals.

In [ ]:
%%time
MODEL_SETS = {
    "BASE": BASE_FEATURES,
    "BASE+B": BASE_FEATURES + BLOCK_B,
    "BASE+B+C": BASE_FEATURES + BLOCK_B + BLOCK_C,
    "ALL": ALL_FEATURES,
}
MODELS = {
    "RandomForest (300, d12)": lambda: RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=42),
    "HistGradientBoosting": lambda: HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.05, max_iter=400, random_state=42),
}

rows = []
for set_name, feature_list in MODEL_SETS.items():
    X = clean_features(exp_df[feature_list])
    for model_name, make_model in MODELS.items():
        model = make_model().fit(X[exp_trainval], y_exp[exp_trainval])
        proba = model.predict_proba(X[exp_test])[:, 1]
        top_decile = proba >= np.quantile(proba, 0.90)

        rows.append({
            "feature_set": set_name,
            "model": model_name,
            "test_precision": round(precision_score(y_exp[exp_test], (proba >= 0.5).astype(int), zero_division=0), 4),
            "test_auc": round(roc_auc_score(y_exp[exp_test], proba), 4),
            # the metric that matters if you only trade your strongest signals
            "precision_top_decile": round(float(y_exp[exp_test][top_decile].mean()), 4),
            "positive_calls": int((proba >= 0.5).sum()),
        })

model_grid = pd.DataFrame(rows).set_index(["feature_set", "model"])
model_grid

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

base_rate = y_exp[exp_test].mean()
width = 0.35
positions = np.arange(len(MODEL_SETS))

for offset, model_name in zip((-width / 2, width / 2), MODELS):
    vals = [model_grid.loc[(s, model_name), "precision_top_decile"] for s in MODEL_SETS]
    bars = ax.bar(positions + offset, vals, width, label=model_name)
    ax.bar_label(bars, fmt="%.3f", fontsize=8)

ax.axhline(base_rate, color="black", ls="--", lw=1, label=f"base rate ({base_rate:.3f})")
ax.set_xticks(positions, list(MODEL_SETS))
ax.set_ylim(0.5, 0.78)
ax.set_ylabel("precision on the top-decile signals")
ax.set_title("Precision when only the strongest 10% of signals are traded")
ax.legend(fontsize=8)
plt.show()

In [ ]:
# which of the engineered features did the strongest model actually lean on?
X_all_feat = clean_features(exp_df[ALL_FEATURES])
best_model = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=42
).fit(X_all_feat[exp_trainval], y_exp[exp_trainval])

NEW_FEATURES = set(BLOCK_A + BLOCK_B + BLOCK_C + BLOCK_D)
importances = (
    pd.DataFrame({"feature": ALL_FEATURES, "importance": best_model.feature_importances_})
    .assign(block=lambda d: np.where(d.feature.isin(NEW_FEATURES), "engineered (Q5)", "homework"))
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("share of total importance held by the", len(NEW_FEATURES), "engineered features:",
      round(importances.loc[importances.block == "engineered (Q5)", "importance"].sum(), 3))
importances.head(20)

### Findings

**1. The honest baseline is 0.578, not 0.628.** Tuning depth on VALIDATION rather than TEST
drops the single tree from Q4's headline 0.628 to **0.5785** (it picks depth 12, not 5). That
5-point gap is the price of the homework's protocol, which selects `max_depth` on the very set
it reports. Everything below is measured the other way, so it is not comparable to Q4 — and it
is the number I would believe.

**2. Of the four blocks, only the cross-sectional one helps a single tree.**

| feature set | TEST precision | TEST AUC | vs. baseline |
| --- | --- | --- | --- |
| BASE | 0.5785 | 0.5400 | — |
| + A macro curve | 0.5362 | 0.5271 | precision −0.042, AUC −0.013 |
| **+ B cross-sectional** | **0.5869** | **0.5467** | **precision +0.008, AUC +0.007** |
| + C own-history | 0.5832 | 0.5378 | precision +0.005, AUC −0.002 |
| + D market regime | 0.5466 | 0.5432 | precision −0.032, AUC +0.003 |
| + ALL blocks | 0.5388 | 0.4827 | precision −0.040, AUC −0.057 |

Block B is the only one positive on both metrics, which says the "everything is absolute, nothing
is relative" diagnosis was the right one — but the size of the win, +0.008 precision, is inside
the noise floor discussed below.

**3. The blocks that wreck a single tree are exactly the ones the ensembles want.** The ALL row
above is the worst in the table, with an AUC of **0.4827 — worse than random**. Yet ALL is the
*best* feature set for both ensembles. The mechanism is visible in how the blocks are built:
A and D are date-level series, identical across all 33 tickers on any given day. A lone tree
splits on them and memorises specific macro regimes from 2000–2021 that simply do not recur in
2021–2025; depth 3 on BASE+D collapses to "always predict up" (31,053 of 31,408 rows called
positive). Three hundred bagged trees with `min_samples_leaf=50` average that memorisation out
and keep the part that generalises.

**4. The uplift is in the ranking, not at the 0.5 threshold.** No ensemble beats the single
tree's best precision-at-0.5 (0.5869), but precision at a fixed threshold compares models that
make very different numbers of calls (19,896 vs 24,472 vs 29,217). Matched on selectivity —
precision on the strongest 10% of signals — the picture inverts:

| | top-decile precision | TEST AUC |
| --- | --- | --- |
| always predict "up" (base rate) | 0.5511 | 0.500 |
| RandomForest on BASE | 0.5632 | 0.5081 |
| HistGradientBoosting on BASE | 0.6065 | 0.5420 |
| HistGradientBoosting on ALL | 0.6552 | **0.5514** |
| **RandomForest on ALL** | **0.7077** | 0.5411 |

**RandomForest on the full engineered set calls 70.8% of its top-decile signals correctly against
a 55.1% base rate — a +15.7pp uplift, of which +14.5pp comes from the engineered features** (the
same model on BASE alone manages only 0.5632). Best AUC is HistGradientBoosting on ALL at 0.5514,
up from 0.5400 for the baseline tree.

**5. So what was actually missing?** Two different things, and the experiment separates them.
The *features* that mattered most were the ones no amount of tuning could have recovered from the
existing columns — relative standing against peers on the same day (B), and market regime (D),
which only pays off once a model can use it without memorising it. But the larger share of the
uplift came from **how the prediction is consumed**: ranking by probability and acting on the top
decile, rather than thresholding a single tree at 0.5. A decision tree emits at most one
probability per leaf, so thresholding throws the ranking away — which is why Q1–Q4's precision
numbers sit in a narrow band no matter what is fed to them.

### The caveat that outranks all of the above

The target overlaps. Two rows one trading day apart share 29 of the 30 days of their forward
window, so the 31,408 TEST rows carry on the order of ~1,000 independent observations. A
precision difference of one or two points — which covers most of the block table above — is
inside that noise, which is exactly why AUC and the top-decile precision are reported
alongside: they move with the whole ranking rather than with a handful of threshold crossings.
Before acting on any of this I would move to purged, embargoed walk-forward CV and report
intervals instead of point estimates.

---
## Summary of answers

Copied into `answers.md` and then into the submission form.

In [ ]:
pd.DataFrame(
    [
        ("1", "max |corr| of a <month>_w<week> dummy", f"{q1_answer}"),
        ("2", "precision of the best new hand rule", f"{q2_answer}"),
        ("3", "TEST rows where only pred5_clf_10 is correct", f"{q3_answer}"),
        ("4", "optimal max_depth", f"{best_max_depth}"),
        ("5", "what data is missing", "free text (see above)"),
    ],
    columns=["Q", "question", "answer"],
).set_index("Q")